# ChemBench Tutorial 2: Train a Baseline Model

This notebook walks through the core ChemBench modeling workflow: load standardized data, split with scaffold partitioning, train a **Random Forest** baseline, and visualize predictions with `ModelEvaluator`.

Random Forest is a strong, interpretable tabular baseline that works well on molecular descriptor features (as included in the processed ESOL file) and requires no GPU.

In [ ]:
from chembench.data.loader import ChemBenchDataLoader
from chembench.data.splitter import DataSplitter

loader = ChemBenchDataLoader("esol")
df = loader.load_data()
target_col = loader.get_targets().columns[0]
smiles_col = "smiles" if "smiles" in df.columns else "mol"

splitter = DataSplitter()
train_df, val_df, test_df = splitter.scaffold_split(
    df,
    smiles_col=smiles_col,
    test_size=0.2,
    val_size=0.1,
)

print(f"Target: {target_col}")
print(f"Train / val / test: {train_df.shape}, {val_df.shape}, {test_df.shape}")

## RandomForestModel

`RandomForestModel` in `chembench.models.baselines.sklearn_models` wraps scikit-learn's `RandomForestRegressor` or `RandomForestClassifier`. It inherits from `BaselineModel`, which provides a uniform API:

- `fit(X_train, y_train)` — train on a numeric feature matrix (`pandas.DataFrame`)
- `predict(X)` — generate predictions
- `evaluate(X_test, y_test)` — return task-appropriate metrics (MSE/MAE/RMSE for regression)

Task type (`regression` vs `classification`) is auto-detected from the target column unless you pass `task_type` explicitly. For ESOL we use the precomputed numeric descriptors (molecular weight, polar surface area, etc.) as features.

In [ ]:
from chembench.models.baselines.sklearn_models import RandomForestModel

drop_cols = [target_col, smiles_col, "Compound ID"]
feature_cols = (
    train_df.drop(columns=[c for c in drop_cols if c in train_df.columns], errors="ignore")
    .select_dtypes(include=["number"])
    .columns.tolist()
)

X_train = train_df[feature_cols]
y_train = train_df[target_col]
X_test = test_df[feature_cols]
y_test = test_df[target_col]

model = RandomForestModel(n_estimators=100, task_type="regression")
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
metrics = model.evaluate(X_test, y_test)
print(metrics)

In [ ]:
from pathlib import Path

from IPython.display import Image, display

from chembench.evaluation.evaluator import ModelEvaluator

evaluator = ModelEvaluator()
plot_path = evaluator.plot_predictions(
    y_true=y_test.values,
    y_pred=y_pred,
    save_dir=Path("tutorial_outputs"),
    model_name="RandomForest_ESOL",
    task_type="regression",
)

print(f"Saved plot to: {plot_path}")
display(Image(filename=str(plot_path)))